# 第 03 章 · 红杯演示查看器

选择 **CPU** 运行时，然后点「运行全部」。Notebook 会自动准备红杯放左盘的小样本。你会看到相机画面、七维动作曲线，以及一个可以用于训练的样本。

第一次运行后，修改「动手试一试」里的参数，再运行参数格和结果格即可。


In [ ]:
# @title 准备实验代码和红杯小样本
import os
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

work = Path("/content/vla-chapter03")
work.mkdir(exist_ok=True)
repo = work / "VLA-Step-by-Step"
source_zip = None  # 用本地代码时，填入上传到 Colab 的源码 ZIP 路径。

def run(*args, **kwargs):
    subprocess.run([str(arg) for arg in args], check=True, **kwargs)

if source_zip is not None:
    repo = work / "VLA-Step-by-Step-zip"
    with ZipFile(source_zip) as archive:
        archive.extractall(repo)
elif repo.exists():
    run("git", "-C", repo, "pull", "--ff-only")
else:
    run("git", "clone", "--depth", "1", "https://github.com/Blue0w/VLA-Step-by-Step.git", repo)

lab = repo / "chapters/03_dataset/lab"
sample_file = lab.parent / "media/libero90-red-mug-left-sample.hdf5"
if not sample_file.is_file():
    raise RuntimeError("找不到第 03 章红杯样本。请重新运行准备格，或上传包含本章样本的源码 ZIP。")
run(sys.executable, "-m", "pip", "-q", "install", "-r", lab / "requirements.txt")
env = os.environ.copy()
env.update(PYTHONPATH=str(repo / "src"), PYTHONIOENCODING="utf-8")


In [ ]:
# @title 动手试一试
# 换成 demo_1 或 demo_2，就能查看同一红杯任务的另一条真实演示。
demo_id = "demo_0"

# t 选的是动作下标。填 None 就选中间一步，填 -2 就选倒数第二个动作。
t = None

# K 是模型每次要预测的动作步数。试试 t = -2 和 K = 4，观察掩码。
K = 1


In [ ]:
# @title 查看轨迹和训练样本
from IPython.display import Image, display

output = work / "trajectory.png"
args = ["--demo-id", demo_id, "--horizon", str(K), "--output", str(output)]
if t is not None:
    args.extend(["--t", str(t)])

result = subprocess.run(
    [sys.executable, str(lab / "demo.py"), *args],
    cwd=repo, env=env, encoding="utf-8", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(result.stdout, end="")
if result.returncode:
    raise RuntimeError("数据实验运行失败，请查看上面的日志。")
display(Image(filename=str(output)))
